# MA9 / MA20 Crossover Backtest

1. Load one asset
2. Calculate MA9 and MA20
3. Walk-forward: buy when MA9 > MA20, sell when MA9 < MA20
4. Start with 1000 EGP
5. Report final value, max drawdown, buy/sell counts
6. Chart it

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed

## 1. Load one asset

In [ ]:
feed = DataFeed.from_dir("data/egx")

SYMBOL = "ABUK"  # change to whichever asset you're assigned

idx = feed.symbols.index(SYMBOL)
dates = pd.to_datetime(feed.dates)
close = feed.close[:, idx]

df = pd.DataFrame({"Date": dates, "Close": close}).dropna().reset_index(drop=True)
df.head()

## 2. Calculate MA9 and MA20

In [ ]:
df["MA9"] = df["Close"].rolling(9).mean()
df["MA20"] = df["Close"].rolling(20).mean()

df = df.dropna().reset_index(drop=True)
df.head()

## 3-5. Walk-forward simulation

Buy when MA9 crosses above MA20 (put all cash into shares).
Sell when MA9 crosses below MA20 (convert shares back to cash).
Starting capital: 1000 EGP.

In [ ]:
INITIAL_CAPITAL = 1000  # EGP

cash = INITIAL_CAPITAL
shares = 0.0
in_position = False

buy_count = 0
sell_count = 0
buy_signals = []
sell_signals = []
portfolio_values = []

for i in range(len(df)):
    price = float(df.loc[i, "Close"])
    ma9 = float(df.loc[i, "MA9"])
    ma20 = float(df.loc[i, "MA20"])
    date = df.loc[i, "Date"]

    if ma9 > ma20 and not in_position:
        shares = cash / price
        cash = 0.0
        in_position = True
        buy_count += 1
        buy_signals.append((date, price))

    elif ma9 < ma20 and in_position:
        cash = shares * price
        shares = 0.0
        in_position = False
        sell_count += 1
        sell_signals.append((date, price))

    portfolio_values.append(cash + shares * price)

df["Portfolio"] = portfolio_values

## 6-8. Results — final value, max drawdown, trade counts

In [ ]:
final_value = df["Portfolio"].iloc[-1]

running_max = df["Portfolio"].cummax()
drawdown_pct = (df["Portfolio"] - running_max) / running_max
max_drawdown_pct = drawdown_pct.min()
max_drawdown_egp = (df["Portfolio"] - running_max).min()

print(f"Asset:                     {SYMBOL}")
print(f"Starting capital:         {INITIAL_CAPITAL:,.2f} EGP")
print(f"Final portfolio value:    {final_value:,.2f} EGP")
print(f"Total return:             {(final_value / INITIAL_CAPITAL - 1):.2%}")
print(f"Max drawdown:             {max_drawdown_pct:.2%}  ({max_drawdown_egp:,.2f} EGP)")
print(f"Number of buy operations:  {buy_count}")
print(f"Number of sell operations: {sell_count}")

## 9. Chart it

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df["Date"], df["Close"], label="Close", alpha=0.6)
axes[0].plot(df["Date"], df["MA9"], label="MA9")
axes[0].plot(df["Date"], df["MA20"], label="MA20")

if buy_signals:
    bx, by = zip(*buy_signals)
    axes[0].scatter(bx, by, marker="^", color="green", s=70, label="Buy", zorder=5)
if sell_signals:
    sx, sy = zip(*sell_signals)
    axes[0].scatter(sx, sy, marker="v", color="red", s=70, label="Sell", zorder=5)

axes[0].set_title(f"{SYMBOL} — Price with MA9/MA20 Crossover Signals")
axes[0].legend()

axes[1].plot(df["Date"], df["Portfolio"], color="purple")
axes[1].axhline(INITIAL_CAPITAL, color="gray", linestyle="--", alpha=0.5, label="Starting capital")
axes[1].set_title("Portfolio Value Over Time (EGP)")
axes[1].set_xlabel("Date")
axes[1].legend()

plt.tight_layout()
plt.show()